In [1]:
import pandas as pd
import numpy as np

In [2]:
logreg_data = pd.read_csv("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_communities_embedding_classified_logreg.csv")
nn_data = pd.read_csv("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_communities_embedding_classified_nn.csv")
verified_communities = pd.read_csv("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/inputs/AI Pattern Project_ Results - Verified Communities.csv")
numerical_cols = [col for col in logreg_data.columns if col not in ['pattern', 'file']]

communities = pd.read_json("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/pattern_classification_verification_results_NN_v2_list.json")
communities['file'] = communities['code file'].apply(lambda x: x.replace('https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/',''))

In [57]:
logreg_data.iloc[7][numerical_cols].index[np.argmax(logreg_data.iloc[7][numerical_cols].values)]

'Reliable, Transparent, & Augmented LLMs'

In [58]:
def get_top_predictions(row):
    return (row[numerical_cols].index[np.argmax(row[numerical_cols].values)], row[numerical_cols].max())

In [76]:
merged_data = {}
verified_files = [file.replace('https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/','') for file in verified_communities['Code file'].tolist()]
for _, row in logreg_data.iterrows():
    file_name = row['file']
    if file_name in verified_files:
        print(f"Skipping verified file: {file_name}")
        continue
    logreg_pred, logreg_proba = get_top_predictions(row)
    
    nn_row = nn_data[nn_data['file'] == file_name].iloc[0]
    nn_pred, nn_proba = get_top_predictions(nn_row)

    code_summary = communities[communities['file'] == file_name]['code summary'].values
    if len(code_summary) > 0:
        code_summary = code_summary[0]        
        merged_data[file_name] = {
            'file': file_name,
            'code summary': code_summary,
            'logreg prediction': logreg_pred,
            'logreg proba': logreg_proba,
            'nn prediction': nn_pred,
            'nn proba': nn_proba,
            'agreement': logreg_pred == nn_pred,
            'agreed pattern': logreg_pred if logreg_pred == nn_pred else ''
        }

Skipping verified file: 3DOD_thesis/cluster_0.py
Skipping verified file: 3DOD_thesis/cluster_1.py
Skipping verified file: 3DOD_thesis/cluster_10.py
Skipping verified file: 3DOD_thesis/cluster_4.py
Skipping verified file: 3DOD_thesis/cluster_5.py
Skipping verified file: 3DOD_thesis/cluster_8.py
Skipping verified file: 3DOD_thesis/cluster_9.py
Skipping verified file: AIlice/cluster_0.py
Skipping verified file: AIlice/cluster_1.py
Skipping verified file: AIlice/cluster_10.py
Skipping verified file: AIlice/cluster_11.py
Skipping verified file: AIlice/cluster_14.py
Skipping verified file: AIlice/cluster_15.py
Skipping verified file: AIlice/cluster_16.py
Skipping verified file: AIlice/cluster_17.py
Skipping verified file: AIlice/cluster_19.py
Skipping verified file: AIlice/cluster_2.py
Skipping verified file: AIlice/cluster_21.py
Skipping verified file: AIlice/cluster_23.py
Skipping verified file: AIlice/cluster_24.py
Skipping verified file: AIlice/cluster_25.py
Skipping verified file: AIlic

In [77]:
predction_df = pd.DataFrame(data=merged_data.values())

In [75]:
predction_df.to_csv("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_communities_embedding_classified_merged_predictions.csv", index=False)

In [78]:
predction_df

,file,code summary,logreg prediction,logreg proba,nn prediction,nn proba,agreement,agreed pattern
0,AIlice/cluster_22.py,This code implements an asynchronous AI agent ...,LLM based Multimodal Generative Prompting,0.963785,LLM based Multimodal Generative Prompting,0.981022,True,LLM based Multimodal Generative Prompting
1,BruteForceAI/cluster_1.py,The code implements an AI pattern centered on ...,Model Abstraction Pattern,0.458667,Model Abstraction Pattern,0.298116,True,Model Abstraction Pattern
2,BruteForceAI/cluster_2.py,This code implements a robust AI pattern for f...,Model Abstraction Pattern,0.840786,Model Abstraction Pattern,0.535428,True,Model Abstraction Pattern
3,BruteForceAI/cluster_3.py,The code implements an AI pattern for automate...,Tool Use for LLMs,0.744664,Tool Use for LLMs,0.609340,True,Tool Use for LLMs
4,CAAFE/cluster_2.py,This code implements an iterative AI pattern f...,LLM Results Evaluation,0.712479,LLM Results Evaluation,0.876375,True,LLM Results Evaluation
...,...,...,...,...,...,...,...,...
1320,viral-clips-crew/cluster_6.py,This code implements a robust pattern for mana...,Preprocessing Text and Numerical Data,0.568947,nan,0.718428,False,
1321,viral-clips-crew/cluster_7.py,This code exemplifies an AI pattern focused on...,Structured Output & Formatting for LLMs,0.637044,Structured Output & Formatting for LLMs,0.519368,True,Structured Output & Formatting for LLMs
1322,web-eval-agent/cluster_1.py,The code implements an **AI Agent Control** pa...,Tool Use for LLMs,0.794168,Tool Use for LLMs,0.567292,True,Tool Use for LLMs
1323,web-eval-agent/cluster_2.py,The provided code primarily consists of utilit...,LLM Results Evaluation,0.351919,nan,0.962233,False,


In [92]:
def stratified_sample_exact(df, group_col, n, random_state=42):
    base_n = n // df[group_col].nunique()

    base = (
        df.groupby(group_col, group_keys=False)
        .apply(lambda x: x.sample(
            n=min(len(x), base_n),
            random_state=random_state
        ))
    )

    remaining = n - len(base)
    if remaining > 0:
        extra = df.drop(base.index).sample(
            n=remaining,
            random_state=random_state
        )
        return pd.concat([base, extra])

    return base

agreed_50_samples = stratified_sample_exact(predction_df[predction_df['agreement'] == True], 'agreed pattern', 50)
disagreed_50_samples = stratified_sample_exact(predction_df[predction_df['agreement'] == False], 'nn prediction', 50)
agreed_50_samples.to_csv("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_communities_embedding_classified_merged_predictions_stratified_50_agreed.csv", index=False)
disagreed_50_samples.to_csv("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_communities_embedding_classified_merged_predictions_stratified_50_disagreed.csv", index=False)

/tmp/ipykernel_124891/2680433670.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(
/tmp/ipykernel_124891/2680433670.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


(50, 8)